# Phase 0 — Donor Structure Audit of cpg0038

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PatrickJReed/cerberus-neuro/blob/main/notebooks/03_donor_audit.ipynb)

**Goal:** Tabulate donor lines per condition, per cell type. Compute imbalance metrics.
Estimate the crop budget for Phase 2. Produce inputs for the Phase 0 audit report.

**Spec:** `docs/superpowers/specs/2026-05-12-argus-cells-design.md`, §3 + §5.
**Plan:** `docs/superpowers/plans/2026-05-12-argus-cells-phase-0-donor-audit.md`.

In [ ]:
!pip install -q --upgrade git+https://github.com/PatrickJReed/cerberus-neuro.git@main

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

from cerberus_neuro.data import build_manifest
from cerberus_neuro.audit import (
    donor_counts_by_condition,
    donor_well_table,
    imbalance_metric,
    crop_budget_estimate,
)

pd.set_option("display.max_rows", 200)

CACHE_DIR = Path("/content/drive/MyDrive/cerberus-neuro/cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Cell 5 — Build the manifest from cpg0038 platemaps + load_data CSVs.
# This is identical to what 01_data_exploration.ipynb does; ~1-3 min.
manifest = build_manifest(cache_dir=CACHE_DIR)
print(f"Manifest rows (sites): {len(manifest):,}")
print(f"Unique wells: {manifest.groupby(['Metadata_Plate', 'Metadata_Well']).ngroups:,}")
print(f"Cell types: {sorted(manifest['Metadata_cell_type'].unique())}")
print(f"Conditions: {sorted(manifest['Metadata_line_condition'].unique())}")
print(f"Donors (line_IDs): {sorted(manifest['Metadata_line_ID'].unique())}")
manifest.head()

In [ ]:
# Cell 6 — Donor counts per condition. Gate check: N >= 3 per condition.
counts = donor_counts_by_condition(manifest)
print("Donor counts per condition:")
for cond, n in counts.items():
    flag = "OK" if n >= 3 else "BLOCKER"
    print(f"  {cond}: {n} donor lines  [{flag}]")

In [ ]:
# Cell 7 — Full per-(cell_type, condition, donor) well-count table.
well_table = donor_well_table(manifest)
print(f"Total (cell_type, condition, donor) groups: {len(well_table)}")
print(well_table.to_string(index=False))

In [ ]:
# Cell 8 — Per-(cell_type, condition) donor-balance CV.
# CV = 0 is perfectly balanced; high CV means one donor dominates.
imbalance = imbalance_metric(well_table)
imb_df = pd.DataFrame(
    [
        {"cell_type": k[0], "line_condition": k[1], "cv": v["cv"], "n_donors": v["n_donors"]}
        for k, v in imbalance.items()
    ]
)
print(imb_df.to_string(index=False))

In [ ]:
# Cell 9 — Crop budget upper-bound at three candidate crops_per_site values.
# Phase 0.5 / Phase 1 yielded ~16k crops at crops_per_site=10. Phase 2 target
# is ~50-100k. Try crops_per_site = 10, 30, 50.
for cps in [10, 30, 50]:
    budget = crop_budget_estimate(manifest, crops_per_site=cps)
    print(
        f"  crops_per_site={cps:>3}: "
        f"sites={budget['n_sites']:,}, "
        f"wells={budget['n_wells']:,}, "
        f"upper-bound crops={budget['max_crops_upper_bound']:,}"
    )